In [1]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig, run_sigGPCCM_experiment
from src.sp_ccm import run_SP_CCM, SP_CCM_iaaft, run_ccm_experiment
from src.iaaft import surrogates

from scipy.stats import ranksums
torch.set_printoptions(sci_mode = False)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



In [3]:
co2_norm = torch.load("data/CO2_vostok_stan_400kyr_timeseries.pt").to(torch.float32)

# Initalise
# g = torch.tensor([0.4, 0.3, 0.2])
g = torch.tensor([0.2, 0.1, 0.1])

true_offset = -2

for t in range(2, co2_norm.shape[0] + 1):
    
    # Co2 is external forcing 
    # This worked: g_next = (g[t] * (1.4 - (1.9 * g[t]) - (0.2 * co2_norm[t - 2])))
    g_next = g_next = (g[t] * (1.4 - (1.9 * g[t]) - (0.3 * co2_norm[t + true_offset])))

    g = torch.concat((g, g_next.unsqueeze(0)))

g_norm = g.sub(g.mean(dim = -1).unsqueeze(-1)).div(g.std(dim = -1).unsqueeze(-1))[0:401]

In [39]:
MAX = 400

fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = 4.8 + co2_norm[0:MAX],
                    mode = 'lines',
                    name = 'd (C02 Vostok)',
                    line_color = "#1E68FF"))

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = 0.7 + g_norm[0:MAX],
                    mode = 'lines',
                    name = 'e',
                    line_color = "#AB3B00",
                    line_dash = "dot"))

fig.update_layout(title = '',
                   xaxis_title = 't',
                   yaxis_title = '')

fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[- 2, MAX])
fig.update_layout(legend = dict(x = 1.0, y = 1.01, bgcolor = "rgba(0,0,0,0)"))

fig.update_layout(autosize = False, width = 800, height = 350)

fig.show()

In [54]:
MAX = 400

fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = 0.8 + g_norm[0:MAX],
                    mode = 'lines',
                    name = 'd (C02 Vostok)',
                    line_color = "#1E68FF",
                    line = dict(width = 5)))

fig.update_layout(title = '',
                   xaxis_title = 't',
                   yaxis_title = '')

fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato", font_size = 20)
fig.update_layout(xaxis_range=[- 5, MAX])
fig.update_layout(legend = dict(x = 1.0, y = 1.01, bgcolor = "rgba(0,0,0,0)"))

fig.update_layout(autosize = False, width = 1600, height = 400)

fig.show()

In [53]:
MAX = 400

fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = 0.8 + co2_norm[0:MAX],
                    mode = 'lines',
                    name = 'd (C02 Vostok)',
                    line_color = "#AB3B00",
                    line = dict(width = 5)))

fig.update_layout(title = '',
                   xaxis_title = 't',
                   yaxis_title = '')

fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato", font_size = 20)
fig.update_layout(xaxis_range=[- 5, MAX])
fig.update_layout(legend = dict(x = 1.0, y = 1.01, bgcolor = "rgba(0,0,0,0)"))

fig.update_layout(autosize = False, width = 1600, height = 400)

fig.show()